# Library and settings

In [ ]:
import os
os.chdir("..")  # Let the wd be outside

In [ ]:
from src.utility import *

import datasets
import duckdb
import gdown
from huggingface_hub import notebook_login, snapshot_download
import json
from kagglehub import dataset_download
import numpy as np
import pandas as pd
import pooch
import tldextract
from typing import Union, List
import zipfile

In [ ]:
RUN_ALL = True

# Data processing

## Datasets hosted on GitHub

In [ ]:
# We will use Pooch. See https://www.fatiando.org/pooch/dev/index.html

# Dict of filename-hash value pairs for datasets.
github_urls = {
    "news-media-reliability_acl2020_corpus.tsv": "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv",
    "news-media-reliability_emnlp18_corpus.tsv": "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv",
    "domain-quality-ratings_domain_ratings.csv": "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv",
    "fakenewscorpus.zip": "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip",
    "fakehealth_release.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json",
    "fakehealth_story.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json",
    "recovery.csv": "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv",
    "politifact_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv",
    "politifact_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv",
    "gossipcop_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv",
    "gossipcop_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv",
    "test2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv",
    "train2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv",
    "val2.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv",
}

github_registry = {
    "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv": "936cc629df930c7caae8b8506fe3843a6b0576c4f4afcd4b371b0156e8a65515",
    "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv": "7f96b15459a23bc5267ca5dc0df66e71d0dc38a28b797b2ec3d228b818427cbf",
    "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv": "1cf331996d880ae0912c64950a5d9aa8101350a380d4f1c8a81bfab663adc8fd",
    "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip": None,  # Skip this hash-check, since it requires unzipping logic
    "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json": "f0c800154e29f1df8599b91c337ae362641083aa44e3c93f3a75186b861dcd1f",
    "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json": "4d9db2ec053888234313e8ef9a8c999fe9b57cbc3dea8c0181c217827256fd7c",
    "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv": "9b63044e77d4a8d9b8e1d283daf63a338ef3dd4c28bd2ac10121759aabc74982",
    "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv": "abe7fe7aad801b1e2ab5fc963cbd0881d9670190262ae3df85e91c3380bd004c",
    "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv": "2500f86a7addca0f59fe8cf089c0f802d3eca6980f53538992b3b0e9687dbae0",
    "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv": "c6932bffadb1230bc61c66bb2ca298f4ef28ce849d268313ad2eed41d18d4c99",
    "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv": "d721e9a8b7e660da05c089691743c33fecb25367074e936b5fab3d85cdcbe79f",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv": "45fac5748e2da08898a4fda0cb02e4c107990a0111b4a6c3aed1956b38a309a2",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv": "63adafa9360f43788fb16499871ff7665143c554fc8a3cce1af846233ce53f07",
    "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv": "a374044e6f73700f749a1bdaf71174356e437c7e6f7ecd4feb52d2fa82820442",
}

github_dog = pooch.create(
    pooch.os_cache("thesis"),
    "https://github.com/",
    registry=github_registry,
    urls=github_urls,
    allow_updates=False,
)

for auto_fname, url in github_dog.urls.items():
    if url.endswith(".zip"):
        github_dog.fetch(url, processor=pooch.Unzip())
    else:
        fname = github_dog.fetch(url)
        print(f"...{fname[-30:]} has hash {pooch.file_hash(fname)}")

## Datasets hosted by HuggingFace, but not as HuggingFace-Datasets

In [ ]:
# Dict of filename-hash value pairs for datasets.
huggingface_urls = {
    "averitec_train.json": "chenxwh/AVeriTeC/resolve/main/data/train.json",
    "averitec_test.json": "chenxwh/AVeriTeC/resolve/main/data/test.json",
    "averitec_dev.json": "chenxwh/AVeriTeC/resolve/main/data/dev.json",
}

huggingface_registry = {
    "chenxwh/AVeriTeC/resolve/main/data/train.json": "ae5eda7c42ddf1695ef185a7ba1bc716928f5adf57103e4f78aae5f9afe00f9c",
    "chenxwh/AVeriTeC/resolve/main/data/test.json": "c62af19526ebc00fdca21ea857282f061326a247ac265c93d55ab6765defbe49",
    "chenxwh/AVeriTeC/resolve/main/data/dev.json": "499793726b4a5406780928a3d9dedc48d6dd53de778f22437d129cacdb08e300",
}

huggingface_dog = pooch.create(
    pooch.os_cache("thesis"),
    "https://huggingface.co/",
    registry=huggingface_registry,
    urls=huggingface_urls,
    allow_updates=False,
)

for auto_fname, url in huggingface_dog.urls.items():
    fname = huggingface_dog.fetch(url)
    print(f"...{fname[-30:]} has hash {pooch.file_hash(fname)}")

## Datasets hosted on HuggingFace Datasets

In [ ]:
druid_plus = datasets.load_dataset(
    "copenlu/druid", "DRUID+", token=HF_HUB_TOKEN, split="train"
)

## Datasets hosted on Kaggle

In [ ]:
covid19_fake_news_path = dataset_download("arashnic/covid19-fake-news")
politifact_fact_check_path = dataset_download(
    "rmisra/politifact-fact-check-dataset"
)
# The saved Kaggle cache path contains a single file.
politifact_fact_check_path = os.path.join(
    politifact_fact_check_path, os.listdir(politifact_fact_check_path)[0]
)

# Remove some unnecessary datafiles
for path in os.listdir(covid19_fake_news_path):
    if "tweets" in path:
        os.remove(os.path.join(covid19_fake_news_path, path))

## Other datasets

In [ ]:
multifc = pooch.retrieve(
    "https://www.copenlu.com/publication/2019_emnlp_augenstein/MultiFC_Codalab.zip",
    known_hash="92aa1563ae21c14b9bb377b468976f774ae3287c94321f752fd785b28134cb4f",
    path=pooch.os_cache("thesis"),
    processor=pooch.Unzip(
        members=[
            "public_data.zip",
            "reference_multi_fc.zip",
            "scoring_multifc.zip",
        ]
    ),
)

mm_covid19_english_news = gdown.download(
    "https://drive.google.com/uc?export=download&id=13dSxwMPrN53YHwvq_D_UPkj6AymMclQW",
    str(
        pooch.os_cache("thesis").joinpath("mm_covid19_english_news.csv")
    ),  # Needs to be string for some reason
    quiet=False,
    resume=True,
)


# This is a nested zipfile, so unzip the rest here
for path in multifc:
    if path.endswith(".zip"):  # This check is only necessarry on repeat runs
        with zipfile.ZipFile(path) as z:
            z.extractall(path + ".unzip")

Be aware that Snopes, PolitiFact and Emergent.Info data must be manually scraped and moved using http://fakenews.research.sfu.ca/#parseWebs !

## Create single Pooch instance for watching all files not managed by KaggleHub or HuggingFaceHub

In [ ]:
registry_path = pooch.os_cache("thesis").joinpath("registry.txt")
pooch.make_registry(pooch.os_cache("thesis"), registry_path, recursive=True)
# Currently stores the registry within the Cache, hence it can become registered within itself. Change this if issues arise.

dog = pooch.create(pooch.os_cache("thesis"), base_url="")
dog.load_registry(registry_path)
# Its registry is more bloated, because it is recursively fetched

# Create a lookup table with more readable filenames
file_aliases = {
    "averitec_dev.json": "chenxwh/AVeriTeC/resolve/main/data/dev.json",
    "averitec_test.json": "chenxwh/AVeriTeC/resolve/main/data/test.json",
    "averitec_train.json": "chenxwh/AVeriTeC/resolve/main/data/train.json",
    "emergent.csv": "emergent_phase1_clean_2018_7_2/emergent_phase1_clean_2018_7_2.csv",
    "domain_ratings.csv": "hauselin/domain-quality-ratings/raw/main/data/domain_ratings.csv",
    "fake_news_corpus.json": "architapathak/FakeNewsCorpus/raw/master/ACL2019_Data.zip.unzip/ACL2019_Data/Fakes700.json",
    "health_release.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthRelease.json",
    "health_story.json": "EnyanDai/FakeHealth/raw/master/dataset/reviews/HealthStory.json",
    "multifc_public_data_dev.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/dev.tsv",
    "multifc_public_data_test.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/test.tsv",
    "multifc_public_data_train.tsv": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/public_data.zip.unzip/train.tsv",
    "multifc_reference_labels.txt": "ec43320311f9d0e79384f2a034e1291f-MultiFC_Codalab.zip.unzip/reference_multi_fc.zip.unzip/labels.txt",
    "news_media_reliability_acl": "ramybaly/News-Media-Reliability/raw/master/data/acl2020/corpus.tsv",
    "news_media_reliability_emnlp": "ramybaly/News-Media-Reliability/raw/master/data/emnlp18/corpus.tsv",
    "politifact.csv": "politifact_phase1_clean_2018_7_3/politifact_phase1_clean_2018_7_3.csv",
    "recovery.csv": "apurvamulay/ReCOVery/raw/master/dataset/recovery-news-data.csv",
    "snopes.csv": "snopes_phase1_clean_2018_7_3/snopes_phase1_clean_2018_7_3.csv",
    "politifact_fact_check_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_fake.csv",
    "politifact_fact_check_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/politifact_real.csv",
    "gossipcop_fact_check_fake.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_fake.csv",
    "gossipcop_fact_check_real.csv": "KaiDMML/FakeNewsNet/raw/master/dataset/gossipcop_real.csv",
    "liar_plus_test.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/test2.tsv",
    "liar_plus_train.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/train2.tsv",
    "liar_plus_val.tsv": "Tariq60/LIAR-PLUS/raw/master/dataset/tsv/val2.tsv",
    "mm_covid19.csv": "mm_covid19_english_news.csv",
}


def dogpath(alias):
    return dog.fetch(file_aliases[alias])

In [ ]:
# Clean up the registry. Ad hoc.
def discardable(path):
    return any(
        [
            "Fakes700/article" in path,
            "reference_multi_fc" in path,
            any(
                [
                    path.endswith(suffix)
                    for suffix in [
                        ".zip",
                        "test.solution",
                        "score.py",
                        "registry.txt",
                    ]
                ]
            ),
        ]
    )


topop = [path for path in dog.registry.keys() if discardable(path)]
for poppable in topop:
    dog.registry.pop(poppable)

# Inspect datasets

## AVeriTeC

In [ ]:
averitec = datasets.load_dataset(
    "json",
    data_files=[dogpath("averitec_dev.json"), dogpath("averitec_train.json")],
    split="train",
)
averitec

In [ ]:
# Add a column with answer types. dataset.flatten() does not work here.
def extract_answer_type_column(row):
    row["answer_types"] = [
        answer["answer_type"]
        for question in row["questions"]
        for answer in question["answers"]
    ]
    return row


averitec = averitec.map(lambda row: extract_answer_type_column(row))
averitec

In [ ]:
averitec_df = averitec.to_pandas()
duplicate_claims_df = averitec_df[averitec_df["claim"].duplicated()]
duplicate_claims_df

In [ ]:
set(averitec_df.claim_date.str[-4:])

- 3,568 rows
- 3.486 unique claims. Multiple reasons for duplicates. Some attribute the same statement to multiple speakers, sometimes hierarchically, e.g. to a person and to their organization. Sometimes, multiple answers i.e. fact-checks exist for the same claim.
- "Speaker" column with names, 44% of which are unique and 22% of which are missing
- "fact_checking_article" column with URLs via web.archive from many different sources. 2368 values.
- "fact_checking_strategies" string-list column
- Labels in {Not Enough Evidence, Supported, Refuted, Conflicting Evidence/Cherrypicking}
- "answer_type" attributes in a nested JSON in the "questions" column.

In [ ]:
averitec_test = datasets.load_dataset(
    "json", data_files=dogpath("averitec_test.json"), split="train"
)
averitec_test

The test set does not contain labels.

In [ ]:
# Let's drop the "Not Enough Evidence"-labeled rows, they don't fit neatly
# downstream.
averitec = averitec.filter(lambda row: row["label"] != "Not Enough Evidence")
averitec

In [ ]:
dir(averitec)

## Breaking! Fake News Dataset

In [ ]:
fake_news_corpus = datasets.load_dataset(
    "json", data_files=dogpath("fake_news_corpus.json"), split="train"
)
fake_news_corpus

- Headlines. "author" column contains both people and organisations. Contains repeat and missing authors. Also contains a "urls" column with the stripped URL!
- "type" column of only 0.97 questionable and 0.03 fake, i.e. <u>a negative dataset!</u>

## Covid Fake News

### Claims

In [ ]:
covid19_fake_claims = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "ClaimFake" in path
    ],
    split="train",
)
covid19_fake_claims = add_binary_label_column(covid19_fake_claims, False)
covid19_real_claims = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "ClaimReal" in path
    ],
    split="train",
)
covid19_real_claims = add_binary_label_column(covid19_real_claims, True)
covid19_claims = datasets.concatenate_datasets(
    [covid19_fake_claims, covid19_real_claims]
)
covid19_claims

- Only 482 rows
- Binary labels
- Source is "news_url" column. This is only a small set of fact-check URLs, hence do not indicate sources.
- <u>Do not use this dataset.</u>

### News

In [ ]:
covid19_fake_news = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "NewsFake" in path
    ],
    split="train",
)
covid19_fake_news = add_binary_label_column(covid19_fake_news, False)
covid19_real_news = datasets.load_dataset(
    "csv",
    data_files=[
        os.path.join(covid19_fake_news_path, path)
        for path in os.listdir(covid19_fake_news_path)
        if "NewsReal" in path
    ],
    split="train",
)
covid19_real_news = add_binary_label_column(covid19_real_news, True)
covid19_news = datasets.concatenate_datasets(
    [covid19_fake_news, covid19_real_news]
)
covid19_news

In [ ]:
if RUN_ALL:
    retrieve_heterogeneously_formatted_years(covid19_news["publish_date"])

- 4504 rows.
- Binary labels.
- 1:5 ratio of false to true labels.
- "type" column for distinguishing social media "post"s from "article"s.
- Contains "fact_check_url" and "news_url({2-5})" for the different URLs cited in the fact-check, i.e. often instagram, reddit, facebook etc.
- Very messy dataset. Missing values, duplicates, unsuccessful scrapes (e.g. scraping login-portals rather than articles).
- Contains archival links for some fact checks.

## Domain Ratings

In [ ]:
domain_ratings = datasets.load_dataset(
    "csv", data_files=dogpath("domain_ratings.csv"), split="train"
)
domain_ratings

- 11,520 rows.
- Ratings from a series of outlets per domain, as well as a derived PC1 value and some statistical values. All ratings are normalized to [0, 1]. Ratings are from NewsGuard, {bias, factuality, final score and minimum} from MBFC, {bias, reliability and derived mean- and minimum} from Ad Fontes Media, >1K Mechanical Turk crowdsourced fact-checkers from the Pennycook paper as well as 60 by professionals jointly labeled as "fact-checkers", the dataset of Lasser et al. <u>which itself is from NewsGuard</u>, and finally, a score derived by assigning 0 to sources from the [Iffy index](https://iffy.news) and otherwise 1, hence potentially containing false positives.
- Not all statistical values apply to each row (presumably based on whether they are motivated given the set of ratings)

## DRUID+

In [ ]:
druid_plus

In [ ]:
druid_plus_df = druid_plus.to_pandas()
druid_plus_df["claimant"].unique()

In [ ]:
sorted(druid_plus_df["claim_date"].str[:7].unique())

In [ ]:
for prefix in (
    druid_plus_df["claim_id"].str.partition("-")[0].str.partition("_")[0].unique()
):
    subset = druid_plus_df[druid_plus_df["claim_id"].str.startswith(prefix)]
    print(prefix)
    print(subset.claimant.unique())
    print(subset.factcheck_verdict.unique(), end="\n\n")

- 48,014 rows.
- "claimant" contains 731 distinct sources in NL. Some claimants are series, e.g. comma-seperated.
- 122 distinct verdicts in "factcheck_verdict" column.
- Some claimants are generic, e.g. "Facebook Post".
- Contains fact-checking entity in "claim_source".
    - The Borderlines subset is weird. Its claimants and labels are geographic locations
    - The CheckYourFact subset contains a small amount of claimants, many of which are generic. Its labels are binary.
    - The FactCheckNI subset is more traditional, but is Ireland-centric. Binary labels.
    - The Factly subset is India-based and contains few unique properly formatted claimants. Labels are ternary, including Half True.
    - The PolitiFact subset is traditional with the same ternary labels.
    - The ClimateFeedback subset contains a small number of claimants. All are labeled False.
    - The HealthFeedback subset contains a small number of claimants. All are labeled False or Half True.
    - The ScienceFeedback subset is traditional. Ternary labels.
    - The SriLankaFactCrescendo subset has mainly generic claimants. All are labeled False.
- `Preliminary thoughts`: Keep PolitiFact and ScienceFeedback subsets.


In [ ]:
chosen_prefixes = ["sciencefeedback", "politifact"]

# This slightly ugly thing just picks out the chosen subset
druid_plus_subset_df = druid_plus_df[
    np.any(
        np.array(
            [
                druid_plus_df["claim_id"].str.startswith(prefix)
                for prefix in chosen_prefixes
            ]
        ),
        axis=0,
    )
]
druid_plus_subset = datasets.Dataset.from_pandas(druid_plus_subset_df)
druid_plus_subset_df

- Resulting subset has 14,654 rows, 310 unique claimants and ternary labels.

## Emergent.Info

In [ ]:
emergent = datasets.load_dataset(
    "csv", data_files=dogpath("emergent.csv"), split="train"
)
emergent

- Only 187 rows
- "fact_tag_phase1" in {false, true, Unverified}
- All rows are dated Jul 3 or missing a date
- The sources only appear as local URLs to articles.
- `Preliminary thoughts`: Skip this dataset. It's too small and hard to work with.

## FakeHealth

In [ ]:
fakehealth = datasets.load_dataset(
    "json",
    data_files=[dogpath("health_release.json"), dogpath("health_story.json")],
    split="train",
)
fakehealth

- 2,296 rows.
- "title"s of the fact-checks and "original_title". Ratings in [0,5]. "Reviewers" column. "source_link". Contains a "news_source" column with the name of the institution e.g. Hospital performing the health-check.

## FakeNewsNet

In [ ]:
fakenewsnet_fake = datasets.load_dataset(
    "csv",
    data_files=dogpath("politifact_fact_check_fake.csv"),
    split="train",
)
fakenewsnet_fake = add_binary_label_column(fakenewsnet_fake, False)

fakenewsnet_real = datasets.load_dataset(
    "csv",
    data_files=dogpath("politifact_fact_check_real.csv"),
    split="train",
)
fakenewsnet_real = add_binary_label_column(fakenewsnet_real, True)

fakenewsnet = datasets.concatenate_datasets(
    [fakenewsnet_fake, fakenewsnet_real]
)
fakenewsnet

- 1056 rows
- From PolitiFact
- A few duplicates
- "news_url" of stories (not cleaned). This is the only source information. 
- Labels in real/fake (I've keyed them to booleans)

## GossipCop

In [ ]:
gossipcop_fake = datasets.load_dataset(
    "csv",
    data_files=dogpath("gossipcop_fact_check_fake.csv"),
    split="train",
)
gossipcop_fake = gossipcop_fake.add_column(
    "label", [False] * len(gossipcop_fake)
)

gossipcop_real = datasets.load_dataset(
    "csv",
    data_files=dogpath("gossipcop_fact_check_real.csv"),
    split="train",
)
gossipcop_real = gossipcop_real.add_column(
    "label", [True] * len(gossipcop_real)
)

gossipcop = datasets.concatenate_datasets([gossipcop_fake, gossipcop_real])
gossipcop

- Same format as previous. Concerns celebrity news.

## Liar-PLUS

In [ ]:
liar_plus = datasets.load_dataset(
    "csv",
    data_files=[
        dogpath("liar_plus_test.tsv"),
        dogpath("liar_plus_train.tsv"),
        dogpath("liar_plus_val.tsv"),
    ],
    split="train",
    delimiter="\t",
    column_names=[
        "ID",
        "Label",
        "Statement",
        "Subjects",
        "Speaker",
        "Speaker's Job Title",
        "State",
        "Party Affiliation",
        # These are cumulative label sums on the speaker including the current
        # statement
        "Barely True Sum",
        "False Sum",
        "Half-true Sum",
        "Mostly True Sum",
        "Pants on Fire Sum",
        "Context",  # Location or context of statement
        "Verdict Justification",
    ],
)
print(set(liar_plus["Party Affiliation"]))
liar_plus

- 12,793 rows.
- "Speaker" are formatted lowercase, dash-seperated; as are labels.
- Contain 24 distinct "Party Affiliation" values in {'republican', 'government-body', 'organization', 'columnist', 'none', 'tea-party-member', None, 'activist', 'constitution-party', 'democratic-farmer-labor', 'business-leader', 'labor-leader', 'liberal-party-canada', 'ocean-state-tea-party-action', 'libertarian', 'county-commissioner', 'education-official', 'democrat', 'independent', 'green', 'newsmaker', 'Moderate', 'talk-show-host', 'state-official', 'journalist'}
- Labels in the PolitiFact Truth-o-meter.

## MisInfo General

In [ ]:
# We have to use snapshot_download to download the dataset together with
# the source metadata without using huggingface-cli.
misinfo_general_dir = snapshot_download(
    repo_id="ioverho/misinfo-general",
    repo_type="dataset",
    token=HF_HUB_TOKEN,
)
misinfo_general_dir

In [ ]:
# The Source metadata is stored in a duckdb database - see
# https://huggingface.co/datasets/ioverho/misinfo-general
metadata_db = duckdb.connect(
    os.path.join(misinfo_general_dir, "metadata.db"),
    read_only=True,
)
print(
    metadata_db.sql(
        """
    SHOW TABLES
    """
    )
)

In [ ]:
for table_name in ["articles", "covid_articles", "sources", "topics"]:
    print(table_name.capitalize())
    print(
        metadata_db.sql(
            f"""
      DESCRIBE {table_name}
      """
        )
    )

### MisInfo General: Sources

In [ ]:
metadata_db.sql(
    f"""
    SELECT *
    FROM sources
    LIMIT 50
    """
)

In [ ]:
# Save the source table as HuggingFace Dataset
# We just want some one representative URL for each source to strip the domain.
misinfo_general_sources = datasets.Dataset.from_pandas(
    metadata_db.sql(
        f"""
    WITH sources_with_representative_urls
    AS (
        SELECT source, MIN(url) representative_article_url
        FROM articles
        GROUP BY source
    )
    SELECT *
    FROM (sources_with_representative_urls NATURAL JOIN sources)
    """
    ).df()
)
misinfo_general_sources

In [ ]:
# Investigate the few articles that have one of 'pseudosci' and 'conspiracy'
# but not both
metadata_db.sql(
    f"""
   SELECT *
   FROM sources
   WHERE (conspiracy IS NULL AND NOT pseudosci IS NULL) OR (NOT conspiracy IS NULL AND pseudosci IS NULL)
   """
)

- Only 488 distinct sources/rows.
- "source" are in NL, lowercase collapsed together with no spaces - <u>however</u>, these don't seem to map directly to domains. For example, the dataset card contains the source "themanchestereveningnews" whose associated domain cuts the "the"-prefix.
- "url" for the FC article.
- "label" in {'left', 'left-center', 'least biased', 'right-center', 'right', 'pro-science', 'satire', 'questionable source', 'conspiracy-pseudoscience'}. These are essentially high-level labels of the source. 34% are Conspiracy-Pseudoscience, 19% are Questionable Source.
- "bias" in {'extreme left', 'left', 'left-center', 'least biased', 'right-center', 'right', 'extreme right'}. Missing from 22%. Otherwise top values are Right (0.21), Left-Center (0.17) and Extreme Right (0.15).
- "factuality" in {'very high', 'high', 'mostly factual', 'mixed', 'low', 'very low'}. Missing from 23%. Top are Low (0.28), Mixed (0.26), High (0.23).
- "country".
- "type" describing the medium, e.g. Website, Newspaper, TV Station... Missing from 5%. Top are Website (0.65), Newspaper (0.12) and Organization/Foundation (0.07). 
- "popularity" in {minimal, medium, high} - <u>can possibly be used for ranking sources!</u>
- "credibility" in {high (0.32), medium (0.1), low (0.45)}.
- "reasoning" column with tags for the reasonings of the label - only for 14% of the data.
- "conspiracy" in {'weak', 'mild', 'moderate', 'strong', 'tin foil hat'} (or NULL) and "pseudosci" in {'weak', 'mild', 'moderate', 'strong', 'quackery'} (or NULL), specifically for sources labeled as Conspiracy-Pseudoscience. Almost but not always coming in pairs (32% have "conspiracy" value, 33% have "pseudosci" value), i.e. seem to always come in pairs. Among tagged datapoints, the labels skew strong. The five situations that do not have pairwise values have "pseudoscience" labels but no "conspiracy" labels.
- "check_date"s which contain clock-time but always at 00:00:00.

## MM_COVID19

In [ ]:
mm_covid19 = datasets.load_dataset(
    "csv", data_files=dogpath("mm_covid19.csv"), split="train"
)
mm_covid19

In [ ]:
# Data Wrangler seems to struggle with this dataset (possible formatting error?), so let's use Pandas.
mm_covid19_df = mm_covid19.to_pandas()
mm_covid19_df[mm_covid19_df["label"] == "fake"]

In [ ]:
# Let's drop the non-statements
mm_covid19_df.dropna(subset=["claim"], inplace=True)

In [ ]:
def _augment_with_canonical_urls_and_authors(row):
    canonical_link = None
    authors = None
    if row["ref_source"]:
        ref_sources_dict = json.loads(row["ref_source"])
        # Some links consist only of whitespace.
        if "canonical_link" in ref_sources_dict.keys() and len(
            ref_sources_dict["canonical_link"].strip()
        ):
            canonical_link = ref_sources_dict["canonical_link"]
        # We leave empty author lists as None for easier dropping in Pandas in
        # the following.
        if "authors" in ref_sources_dict.keys() and len(
            ref_sources_dict["authors"]
        ):
            authors = ref_sources_dict["authors"]
    row["canonical_link"] = canonical_link
    row["authors"] = authors
    return row


mm_covid19 = mm_covid19.map(_augment_with_canonical_urls_and_authors)

In [ ]:
mm_covid19_df = mm_covid19.to_pandas()
# Drop rows with neither authors nor canonical URLs
mm_covid19_df.dropna(
    subset=["canonical_link", "authors"], how="all", inplace=True
)
mm_covid19_df[mm_covid19_df["label"] == "fake"]

In [ ]:
mm_covid19 = datasets.Dataset.from_pandas(mm_covid19_df)

mm_covid19_authors = extract_set_of_authors_from_list_column(
    mm_covid19, "authors"
)
print(len(mm_covid19_authors))
mm_covid19_authors

- 7423 rows
- Covid19-related
- 46% missing claims which are tweets of statements or nonverifiable claims, probably for dataset balance.
- Fact-check URLs, fact-checking organization in "agency"
- Labels in {real, fake}
- The direct URL to the claim can be found in "ref_source_url", but this is often archive entries or URL-shortenings. The canonical URL together with an optional list of authors can be extracted from a nested JSON object in "ref_source".

## MultiFC

In [ ]:
multifc = datasets.load_dataset(
    "csv",
    data_files=[
        dogpath("multifc_public_data_dev.tsv"),
        dogpath("multifc_public_data_train.tsv"),  # The test set does not
        # contain labels
    ],
    delimiter="\t",
    split="train",
    column_names=[
        "claimid",
        "claim",
        "label",
        "claim url",
        "reason",
        "category",
        "speaker",
        "checker",
        "tags",
        "article title",
        "publish date",
        "claim date",
        "claim entities",
    ],  # Derived from the paper, but their exact ordering was manually
    # achieved
)
multifc

In [ ]:
if RUN_ALL:
    print(
        f"Claim dates: {retrieve_heterogeneously_formatted_years(multifc['claim date'])}"
    )
    print(
        f"Publish dates: {retrieve_heterogeneously_formatted_years(multifc['publish date'])}"
    )

- Nearly 35,000 rows
- Heterogenous dataset; some columns have differently formatted data, including the "Claim URL" and "Publish Date" columns
- Claims, heterogenous labels (3590 distinct), URLs, textual label reason, "Speaker" row, "Checker" row (only 354 distinct), titles
- Many missing elements, some duplicates

## NewsMediaReliability

In [ ]:
news_media_reliability_emnlp = datasets.load_dataset(
    "csv",
    data_files=dogpath("news_media_reliability_emnlp"),
    delimiter="\t",
    split="train",
)

news_media_reliability_emnlp

- URLs, factuality in {low 0.24, mixed 0.25, high 0.51}, bias in {(extreme) (center)-{left, right}}
- 1066
- All sources are from MBFC

In [ ]:
news_media_reliability_acl = datasets.load_dataset(
    "csv",
    data_files=dogpath("news_media_reliability_acl"),
    delimiter="\t",
    split="train",
)

news_media_reliability_acl

- A cleansed version of the EMNLP dataset. The authors argue that the center-* labels are inconsistent and therefore the given sources were removed. Furthermore, the extreme labels are collapsed into the regular ones. 
- Hence has bias in {right 0.41, center 0.32, left 0.28} and factuality in {high 0.53, mixed 0.29, low 0.19}
- 859 rows
- One missing reference

Preliminary thoughts: Keep the EMNLP dataset for now. It's a superset, and the extra labels are chosen by MBFC.

## PolitiFact Fact Check Dataset

In [ ]:
politifact_fact_check = datasets.load_dataset(
    "json",
    data_files=politifact_fact_check_path,
    split="train",
)
politifact_fact_check

In [ ]:
if RUN_ALL:
    print(
        f"Statement years: {retrieve_heterogeneously_formatted_years((politifact_fact_check['statement_date']))}"
    )
    print(
        f"Fact-check years: {retrieve_heterogeneously_formatted_years((politifact_fact_check['factcheck_date']))}"
    )

- 21,152 rows.
- Uses the PolitiFact Truth-o-meter.
- "statement_originator" in NL; 4565 distinct.
- "statement_source", being source medium or context, such as speech, campaign, news, etc. (13 distinct).
- "factchecker" (NL).
- "factcheck_analysis_link", i.e. the fact check URL.
- Skews toward negative labels, i.e. lies.
- Dated to a broad timespan in the 2000s.

## ReCoVery

In [ ]:
recovery = datasets.load_dataset(
    "csv", data_files=dogpath("recovery.csv"), split="train"
)
recovery

In [ ]:
recovery_authors = extract_set_of_authors_from_list_column(recovery, "author")
print(len(recovery_authors))
recovery_authors

- 2029 rows
- Unnamed column which seems to be equivalent to the "news_id" column.
- "url", "publisher" in NT, "author" column with array values, often empty, "title", political_bias in (Extreme) (Center-) {left, right}, binary reliability score
- Labels are derived from MBFC and NewsGuard. From the paper: "A news site is reliable if its NewsGuard score is greater than 90, and its factual reporting on MBFC is very high or high. A news site is unreliable if its NewsGuard score is less than 30, and its factual reporting on MBFC is below mixed".
- Hence <u>this is an extreme dataset</u>; middling values do not appear.
- Articles are (all?) published in 2020
- 82% of articles from the US, 10% from Russia
- Some missing values and duplicates

## Snopes

In [ ]:
snopes = datasets.load_dataset(
    "csv", data_files=dogpath("snopes.csv"), split="train"
)
snopes

In [ ]:
set(snopes["fact_rating_phase1"])

In [ ]:
if RUN_ALL:
    retrieve_heterogeneously_formatted_years(snopes["article_date_phase1"])

- 37,957 rows of which 5,640 are unique.
- Again, URLs are to sources used and/or claims on e.g. Twitter, hence this is probably unusable.

# Comparing/merging datasets

The following table is sorted in descending order by the following key: Distinct sources, lists of authors, domain URLs, article URLs, other.

| Name | Num. Distinct sources | Origin(s) | Source designation | Labels | Other useful attributes | Date^ | Dirty?~ | Comments |
| ---| --- | --- | --- | ---| --- | --- | --- | --- |
| Domain Ratings | 11,520 | NewsGuard, MBFC, Ad Fontes Media, crowdsourcing, 60 professional fact-checkers, the Iffy index | Domain | Multiple kinds of scores, mostly in [0, 1] | Statistically derived scores, chiefly their custom PC1 score | Dataset: 2023 | False | The Iffy score is 1 if a domain does not appear on the Iffy index, else 0, hence it can have "false positives" |
| PolitiFact Fact Check | 4565 | PolitiFact | NL | Truth-o-meter | Fact-checker | Data: Statements from 2000 + 2002 + 2007-2022, FCs from 2007-2022 | True | Labels skew negative. |
| MultiFC | 32,628 URLs, 3997 Speakers | Heterogenous | Claim URL, Speaker (NL)| Heterogenous (116 distinct) || Data: Claims from 1995 + 2007-2018, Fact Checks from 1995-2018 | True | Very dirty and heterogeneous |
| Liar-PLUS | 3309 | PolitiFact | Source in normalized lower-case dash-seperated NL | Truth-o-meter | Speaker's job title, cumulative sums of scores per source | Dataset: 2018 | True ||
| AVeriTeC| 1571 speakers, 701 reporting sources | Heterogenous | Speaker (NL) + Reporting source (NL) + URL | {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence} | | Data: 1919-1922 + 2008-2009 + 2011-2020 | False | Each element may have any combination of speaker, reporting source and URL |
| NewsMediaReliability-EMNLP| 1066 | MBFC | (Normalized) Domain | {low, mixed, high} | bias (MBFC style) | Dataset: 2019| False | |
| NewsMediaReliability-ACL| 859| MBFC | (Normalized) Domain  | {low, mixed, high} | bias (MBFC style, extremes collapsed) | Dataset: 2021| True| Cleansed subset of EMNLP |
| MisInfo General: Sources | 488 | MBFC | Lowercase no-space NL | Factuality in {'very high', 'high', 'mostly factual', 'mixed', 'low', 'very low'}, Credibility in {high (0.32), medium (0.1), low (0.45)} | MBFC style bias, high level "label" of source, popularity, reasoning (of label), conspiracy and pseudosci | FCs in 2023-2024 | True | Source names map often but not always to URLs |
| FakeHealth | 462 sources, 2295 URLs | HealthNewsReview.org | NL + Domain | [0, 5] (higher better) | (Not) Satisfactory answer JSON | Dataset: 2020 | False | Health-related news |
| DRUID+ (subset) | 310 sources | PolitiFact, Science Feedback | NL | {False, Half True, True} || Data: 2013-2024 | False ||
| Breaking! Fake News Dataset | 209| Own| Author (NL, name or org) + Domain | {questionable, fake} || Dataset: 2019| True | Negative dataset, only bad sources |
| ReCoVery | 55 publishers, ~1086 authors | MBFC and NewsGuard | Publishers in NL, URLs, arrays of authors in NL | {0, 1} mapped from NewsGuard and MBFC | MBFC style bias | Data: 2020 | True | Concerns the Covid pandemic. Contains only datapoints with extreme labels: From the paper: "A news site is reliable if its NewsGuard score is greater than 90, and its factual reporting on MBFC is very high or high. A news site is unreliable if its NewsGuard score is less than 30, and its factual reporting on MBFC is below mixed". |
| MM-COVID19 | 1845 URLs, ~400 authors | 38 distinct | Direct URLs and lists of authors in NL | {real, fake} || Dataset: 2020 | True | Covid-related. Fairly dirty.|
| FakeNewsNet | 967 URLs | PolitiFact | URL | Binary || Dataset: 2017 | True ||
| Emergent.Info | 165 URLs | Emergent.Info | URL | {false, true, Unverified} || Dataset: 2018 | True ||
| PolitiFact | ?< 73,766 URLs | PolitiFact | Only appear in NL in FC articles | Truth-o-meter || Data: 2007-2018 | False | Sources would have to be extracted from NL. |
| Snopes | ?< 37,958 URLs | Snopes | Only appear in NL in FC articles | {'correct attribution', 'false', 'legend', 'misattributed', 'miscaptioned', 'mixture', 'mostly false', 'mostly true', 'outdated', 'scam', 'true', 'unproven'} || 1996-2018 | False | Similar to the PolitiFact dataset |
| GossipCop | 20,691 URLs | GossipCop | URL | Binary || Dataset: 2017 | True | Concerns celebrity news |
| Covid Fake News: News | ?< 4,505 URLs | Heterogeneous | URLs used in FCs are included, but they may concern the news story itself or a person sharing it | Binary | type in {post, article, NULL} | 2016, 2018-2020 | True | Very dirty. Sources are hard to retrieve from the URLs |
| Covid Fake News: Claims | None | Heterogeneous | None | Binary | Dataset: 2020 | False | No sources. Concerns the Covid pandemic. Unusable. |

~ Duplicates or missing key values  
^ Explicit from data when possible, else dataset publication date. Loosely formatted dates are parsed in a generic manner and hence may be incorrect.

### Attempting to extract and compare differently formatted sets of sources

## NL

In [ ]:
# Let's see how nice these sources look with dashes replaced by space
def _replace_dashes_with_space_in_dataset(col_name, row):
    if row[col_name]:
        row[col_name] = row[col_name].replace("-", " ")
    return row


liar_plus = liar_plus.map(
    lambda row: _replace_dashes_with_space_in_dataset("Speaker", row)
)

liar_plus["Speaker"]

### Sets

In [ ]:
def _num_unique_source_contributions(key: str, superdict: dict):
    """
    Count how many unique sources each dataset contributes with. This is not an optimal generic function for this purpose, but it works for this context.
    """
    assert key in superdict.keys()
    remaining_source_dicts = [
        source_dict for (name, source_dict) in superdict.items() if name != key
    ]
    sources_sans_target = set().union(*remaining_source_dicts)
    return len(superdict[key].difference(sources_sans_target))


def _print_num_unique_source_contributions_per_item(superdict):
    print("\nUnique sources contributed per dataset:")
    for dataset_name in superdict.keys():
        print(
            f"{dataset_name}: {_num_unique_source_contributions(dataset_name, superdict)}"
        )


# Enforce lowercase on all lists for completeness
politifact_fact_check_nl_sources = set(
    str_list_to_lowercase(politifact_fact_check["statement_originator"])
)

liar_plus_nl_sources = set(str_list_to_lowercase(liar_plus["Speaker"]))

averitec_nl_sources = set(
    str_list_to_lowercase(averitec["speaker"] + averitec["reporting_source"])
)

fakehealth_nl_sources = set(str_list_to_lowercase(fakehealth["news_source"]))

fake_news_corpus_nl_sources = set(
    str_list_to_lowercase(fake_news_corpus["authors"])
)
recovery_nl_sources = set(str_list_to_lowercase(recovery["publisher"])).union(
    {author.lower() for author in recovery_authors}
)
multifc_nl_sources = set(str_list_to_lowercase(multifc["speaker"]))

mm_covid19_sources = {author.lower() for author in mm_covid19_authors}

druid_plus_sources = set(str_list_to_lowercase(druid_plus_subset["claimant"]))

all_nl_source_sets = {
    "PolitiFact Fact Check": politifact_fact_check_nl_sources,
    "Liar-PLUS": liar_plus_nl_sources,
    "AVeriTeC": averitec_nl_sources,
    "FakeHealth": fakehealth_nl_sources,
    "Fake News Corpus": fake_news_corpus_nl_sources,
    "ReCoVery": recovery_nl_sources,
    "MultiFC": multifc_nl_sources,
    "MM-COVID19": mm_covid19_authors,
    "DRUID+": druid_plus_sources,
}

# Note that a few of these natural language sources are in fact domains,
# and in general, they are somewhat noisy, e.g. contains entries such as
# "-the telegraph and the daily star". Hence you should probably run
# further cleaning if you are going to concretely use this downstream.
print("NL sources:")
_print_num_unique_source_contributions_per_item(all_nl_source_sets)

nl_sources = set().union(*all_nl_source_sets.values())
print("\nNum. sources:", len(nl_sources))
# Note that the number of distinct sources != the sum of contributions,
# because this count of contributions only concerns sources which are
# unique to a single dataset.

#### Investigate and compare labels to decide on subsets of datasets to keep

Let's take a look at all the labels we have:

- [0, 1]
- Truth-o-meter
- Heterogenous
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence}
- {low, mixed, high}
- Factuality in {very high, high, mostly factual, mixed, low, very low}
- Credibility in {high, medium, low}
- [0, 5]
- {questionable, fake}
- Binary
- {false, true, unverified}

All labels except the following can at least implicitly be mapped to a scalar: 

- {false, true unverified}
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence},
- Heterogenous 

All of those except the last can at least partially implicitly be mapped to a scalar.  

Further, let us consider which of the labels come from dominant datasets.

- [0, 1] from Domain Ratings makes up almost all domains
- Truth-o-meter comes from PolitiFact Fact Check
- MultiFC contributes the large heterogeneous set of labels
- {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence} comes from AVeriTeC

This provokes the question of whether we can maybe pick out a useful subset of labels from MultiFC.

In [ ]:
nl_sources_sans_multifc = [
    source_set
    for (name, source_set) in all_nl_source_sets.items()
    if name != "MultiFC"
]
unique_multifc_nl_sources = multifc_nl_sources.difference(
    set().union(*nl_sources_sans_multifc)
)
multifc_sources_df = multifc.to_pandas()
unique_multifc_sources_df = multifc_sources_df[
    multifc_sources_df["speaker"].str.lower().isin(unique_multifc_nl_sources)
]

unique_multifc_sources_df.groupby("label")["speaker"].nunique().sort_values(
    ascending=False
)

In [ ]:
# Given these observations, it's well-motivated to just extract the regular
# PolitiFact entries.

# We can retrieve the Truth-o-meter labels from this dataset
politifact_truth_o_meter = set(
    [
        s.replace("-", " ")
        for s in politifact_fact_check["verdict"]
        if s != "half-true"
    ]
)

speakers_from_multifc_from_politifact = set(
    str_list_to_lowercase(
        unique_multifc_sources_df[
            unique_multifc_sources_df["label"].isin(politifact_truth_o_meter)
        ]["speaker"].values
    )
)

speakers_from_multifc_from_politifact

Another remaining question is whether we can map the AVeriTeC labels: {Refuted, Conflicting Evidence/Cherrypicking, Supported, Not Enough Evidence}. At first pass, Refuted may equal 0, Conflicting Evidence/Cherrypicking may equal something like 0.5, Supported may equal 1, and Not Enough Evidence may be mapped to something like 0.3 or dropped from the dataset.

In [ ]:
# Supported + Refuted make up 2355/2792 speaker/label pairs.
averitec_df.groupby("label").count()

Hence for now, we might just expect to map Refuted/Supported and drop remaining rows.

In [ ]:
averitec_df = averitec_df[averitec_df["label"].isin(["Refuted", "Supported"])]
averitec_df

### Datasets

Create dataframe that keeps label and provenance information.

In [ ]:
nl_source_df = pd.DataFrame(
    columns=["id", "source_name", "label", "source_dataset"]
)


def merge_df_with_source_df(
    df,
    target_df,
    source_name_col: str,
    label_col: Union[str, List[str]],
    dataset_name: str,
    bias_col: str = None,
):
    idx_start = len(target_df)
    idx_end = idx_start + len(df)
    if isinstance(label_col, List):
        df["label"] = df.apply(lambda row: {col: row[col] for col in label_col}, axis=1)
        if bias_col:
            rename_mapping = {
                source_name_col: "source_name",
                bias_col: "bias",
                "label": "label",
            }
    elif bias_col:
        rename_mapping = {
            source_name_col: "source_name",
            label_col: "label",
            bias_col: "bias",
        }
    else:
        rename_mapping = {source_name_col: "source_name", label_col: "label"}
    df = df.rename(columns=rename_mapping)
    df = df[rename_mapping.values()]
    df["source_dataset"] = dataset_name
    df["id"] = np.arange(idx_start, idx_end)
    if not bias_col:
        df["bias"] = None
    return pd.concat([target_df, df])


nl_source_df = merge_df_with_source_df(
    politifact_fact_check.to_pandas(),
    nl_source_df,
    "statement_originator",
    "verdict",
    "PolitiFact Fact Check",
)

nl_source_df = merge_df_with_source_df(
    liar_plus.to_pandas(), nl_source_df, "Speaker", "Label", "Liar-PLUS"
)

# AveriTeC is the one dataset with two useful source columns. Let's split it. The merged DataFrame will be cleaned later.
averitec_speakers = averitec.to_pandas()[["speaker", "label"]]
nl_source_df = merge_df_with_source_df(
    averitec_speakers, nl_source_df, "speaker", "label", "AVeriTeC"
)
averitec_reporting_sources = averitec.to_pandas()[
    ["reporting_source", "label"]
]
nl_source_df = merge_df_with_source_df(
    averitec_reporting_sources,
    nl_source_df,
    "reporting_source",
    "label",
    "AVeriTeC",
)

nl_source_df = merge_df_with_source_df(
    fakehealth.to_pandas(), nl_source_df, "news_source", "rating", "FakeHealth"
)

nl_source_df = merge_df_with_source_df(
    fake_news_corpus.to_pandas(),
    nl_source_df,
    "authors",
    "type",
    "FakeNewsCorpus",
)

nl_source_df = merge_df_with_source_df(
    recovery.to_pandas(),
    nl_source_df,
    "publisher",
    "reliability",
    "ReCoVery",
    "political_bias",
)

recovery_authors_dataset = extract_dataset_with_authors_from_list_column(
    recovery, "author"
)
nl_source_df = merge_df_with_source_df(
    recovery_authors_dataset.to_pandas(),
    nl_source_df,
    "author",
    "reliability",
    "ReCoVery",
    "political_bias",
)

# Extract the subset of MultiFC we decided to keep.
multifc_politifact_factscan_subset = unique_multifc_sources_df[
    unique_multifc_sources_df["label"].isin(politifact_truth_o_meter)
]

nl_source_df = merge_df_with_source_df(
    multifc_politifact_factscan_subset,
    nl_source_df,
    "speaker",
    "label",
    "MultiFC",
)

nl_source_df = merge_df_with_source_df(
    mm_covid19.to_pandas(), nl_source_df, "agency", "label", "MM-COVID19"
)

mm_covid19_authors_dataset = extract_dataset_with_authors_from_list_column(
    mm_covid19, "authors"
)
nl_source_df = merge_df_with_source_df(
    mm_covid19_authors_dataset.to_pandas(),
    nl_source_df,
    "authors",
    "label",
    "MM-COVID19",
)

nl_source_df = merge_df_with_source_df(
    druid_plus_subset_df,
    nl_source_df,
    "claimant",
    "factcheck_verdict",
    "DRUID+"
)

nl_source_df["label"] = nl_source_df["label"].astype(str)

# Normalize source designation before saving the dataset.
nl_source_df["source_name"] = (
    nl_source_df["source_name"].str.lower().str.strip()
)

nl_sources_dataset = datasets.Dataset.from_pandas(nl_source_df).remove_columns(
    ["__index_level_0__"]
)
nl_sources_dataset

In [ ]:
nl_sources_dataset.to_pandas()

In [ ]:
nl_sources_dataset.to_pandas()["source_dataset"].unique()

Time to aggregate the datasets. Downstream, we want to query WikiData for each source only once. Hence we will do the following: Each source shall have a label column with a dict which maps each source dataset key with a list of ratings from that dataset to the source.

In [ ]:
def aggregate_df_entries(
    df,
    source_col_name="source_name",
    source_dataset_col_name="source_dataset",
    label_col_name="label",
):
    """
    There is some very hairy Pandas logic here. The point is that each label
    given to the same source in the same dataset is aggregated in a list (1),
    and each such list is then collected in a dict whose key is the name of the
    source dataset (2).
    """
    # (1)
    df_grouped_by_name_and_dataset = (
        df.groupby([source_col_name, source_dataset_col_name])[label_col_name]
        .apply(list)
        .reset_index()
    )
    # (2)
    df_aggregated = (
        df_grouped_by_name_and_dataset.groupby(source_col_name)
        .apply(
            lambda row: dict(zip(row[source_dataset_col_name], row[label_col_name])),
            include_groups=False,
        )
        .reset_index(name="dataset_label_map")
    )
    # Recover the bias column
    return df_aggregated.merge(df.groupby(source_col_name).first(), on=source_col_name)


nl_source_df_aggregated = aggregate_df_entries(nl_source_df)
# Recover the bias column
nl_source_df_aggregated["id"] = np.arange(len(nl_source_df_aggregated))

## Domains

### Sets

In [ ]:
domain_ratings_domains = set(domain_ratings["domain"])
news_media_reliability_emnlp_domains = set(
    news_media_reliability_emnlp["source_url_normalized"]
)
fake_news_corpus_domains = set(fake_news_corpus["urls"])

Domains extracted from URLs

In [ ]:
fakenewsnet_domains = extract_set_of_domains_from_urls(fakenewsnet["news_url"])
emergent_domains = extract_set_of_domains_from_urls(
    emergent["original_url_phase1"]
)
gossipcop_domains = extract_set_of_domains_from_urls(gossipcop["news_url"])
misinfo_general_domains = extract_set_of_domains_from_urls(
    misinfo_general_sources["representative_article_url"]
)
mm_covid19_domains = extract_set_of_domains_from_urls(
    mm_covid19["canonical_link"]
)

In [ ]:
all_domain_sets = {
    # Pure domains
    "Domain Ratings": domain_ratings_domains,
    "News Media Reliability EMNLP": news_media_reliability_emnlp_domains,
    "Fake News Corpus": fake_news_corpus_domains,
    # Extracted domains
    "FakeNewsNet": fakenewsnet_domains,
    "Emergent.Info": emergent_domains,
    "GossipCop": gossipcop_domains,
    "MisInfo General": misinfo_general_domains,
    "MM-COVID19": mm_covid19_domains,
}

domains = set().union(*all_domain_sets.values())

print("Domains:")
_print_num_unique_source_contributions_per_item(all_domain_sets)

print("\nNumber of unique domains:", len(domains))

### Datasets

Create domain dataset that keeps label and dataset information.

In [ ]:
news_media_reliability_emnlp.to_pandas()


In [ ]:
domain_df = pd.DataFrame(
    columns=["id", "source_name", "label", "source_dataset"]
)

# NOTE: Currently running with the PC1 ratings. This dataset contains multiple ratings per domain.
domain_df = merge_df_with_source_df(
    domain_ratings.to_pandas(), domain_df, "domain", "pc1", "Domain Ratings"
)

# Stick with the unnormalized URLs. That helps with WikiData crossreferencing later.
domain_df = merge_df_with_source_df(
    news_media_reliability_emnlp.to_pandas(),
    domain_df,
    "source_url_normalized",
    "fact",
    "News Media Reliability",
    "bias",
)

domain_df = merge_df_with_source_df(
    fake_news_corpus.map(
        lambda row: extract_domain_from_url_in_row(row, "urls")
    ).to_pandas(),
    domain_df,
    "urls",
    "type",
    "FakeNewsCorpus",
)

domain_df = merge_df_with_source_df(
    fakenewsnet.map(
        lambda row: extract_domain_from_url_in_row(row, "news_url")
    ).to_pandas(),
    domain_df,
    "news_url",
    "label",
    "FakeNewsNet",
)

domain_df = merge_df_with_source_df(
    emergent.map(
        lambda row: extract_domain_from_url_in_row(row, "original_url_phase1")
    ).to_pandas(),
    domain_df,
    "original_url_phase1",
    "fact_tag_phase1",
    "Emergent.Info",
)

domain_df = merge_df_with_source_df(
    gossipcop.map(
        lambda row: extract_domain_from_url_in_row(row, "news_url")
    ).to_pandas(),
    domain_df,
    "news_url",
    "label",
    "GossipCop",
)

domain_df = merge_df_with_source_df(
    misinfo_general_sources.map(
        lambda row: extract_domain_from_url_in_row(
            row, "representative_article_url"
        )
    ).to_pandas(),
    domain_df,
    "representative_article_url",
    ["factuality", "conspiracy", "pseudosci"],
    "MisInfo General",
    "bias"
)

domain_df = merge_df_with_source_df(
    mm_covid19.map(
        lambda row: extract_domain_from_url_in_row(row, "canonical_link")
    ).to_pandas(),
    domain_df,
    "canonical_link",
    "label",
    "MM-COVID19",
)

In [ ]:
domain_df

At this point, let's quickly move domains that are easily spotted in the NL source set.

In [ ]:
# Ad hoc URL filtering. Not thorough.
@np.vectorize
def _is_probably_url(source_name: str) -> bool:
    return bool(
        source_name is not None
        and (domain_info := tldextract.extract(source_name)) is not None
        and (domain_info.subdomain or domain_info.suffix)
        and " " not in source_name
        and "." in source_name
    )


urls_from_nl_sources = nl_source_df_aggregated[
    _is_probably_url(nl_source_df_aggregated["source_name"].values)
].drop(columns=["dataset_label_map"])

domain_df = pd.concat([domain_df, urls_from_nl_sources])


# Return the subset of nl_sources_dataset that isn't in
# urls_from_nl_sources. Seems faster than built-in mapping-type
# filtering function. Uncertain if there is a more elegant way to do
# this.
mask = np.full_like(nl_source_df_aggregated["id"], fill_value=True, dtype=bool)
mask[urls_from_nl_sources["id"]] = False
nl_source_df_aggregated = nl_source_df_aggregated[mask]
# NOTE: When converting the dataset label map from Pandas to HuggingFace
# datasets, it enforces a consistent schema by adding datasetname-None
# pairs for each dataset that a source does not appear in. This makes it
# more computationally consistent but less readable, hence some local
# view should be used when investigating the set of maps.

nl_source_df_aggregated["dataset_label_map"] = nl_source_df_aggregated["dataset_label_map"].astype("string")
nl_sources_dataset = datasets.Dataset.from_pandas(nl_source_df_aggregated)
nl_sources_dataset.save_to_disk(os.path.join(DATA_DIR, "nl_sources_dataset"))
nl_sources_dataset.to_pandas()

Finish aggregating the domain dataset.

In [ ]:
# Be aware that the domains from Domain Ratings and News Media
# Reliability are incompletely formatted vis-a-vis WikiData URL
# entities, so an experimental approach must be adopted when
# cross-referencing those.
domain_df_aggregated = aggregate_df_entries(domain_df)

# A few newlines have snuck in which cause problems when forming queries later
domain_df_aggregated["dataset_label_map"] = (
    domain_df_aggregated["dataset_label_map"].astype(str).replace("\n", "")
)
domain_df_aggregated.reset_index(inplace=True)
domain_df_aggregated["label"] = domain_df_aggregated["label"].astype(str)

domain_df_aggregated
# domain_dataset = datasets.Dataset.from_pandas(domain_df_aggregated)
# domain_dataset.save_to_disk(os.path.join(DATA_DIR, "domain_dataset"))
# domain_dataset